# Sensitivity Analysis: Tornado Diagram

This notebook demonstrates **parameter sensitivity analysis** using our standard groundwater model.

**Key concept**: Not all parameters are equally important. Sensitivity analysis identifies which parameters most influence model predictions, helping us:
1. Focus calibration efforts on sensitive parameters
2. Understand which uncertainties matter most
3. Prioritize data collection

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import matplotlib.animation as animation
from IPython.display import HTML

# Import common setup
from calibration_common import (
    X_DOMAIN, L, H0, H1, K_TRUE, R_TRUE,
    X_OBS_STANDARD, H_OBS_STANDARD,
    gw_model_1d
)

plt.rcParams['font.size'] = 11
plt.rcParams['figure.facecolor'] = 'white'

## The Model

We use the same 1D confined aquifer model as the other demos:

$$h(x) = h_0 + (h_1 - h_0)\frac{x}{L} + \frac{R}{K} L \frac{x}{L}\left(1 - \frac{x}{L}\right)$$

For sensitivity analysis, we consider these parameters:
- $K$ = Hydraulic conductivity [m/d]
- $R$ = Recharge rate [m/d]
- $h_0$ = Left boundary head [m]
- $h_1$ = Right boundary head [m]
- $L$ = Domain length [m]

In [ ]:
def gw_model_full(K, R, h0, h1, L, x=None):
    """
    Full groundwater model with all parameters explicit.
    """
    if x is None:
        x = np.linspace(0, L, 101)
    x = np.asarray(x)
    xi = x / L
    h = h0 + (h1 - h0) * xi + (R / K) * L * xi * (1 - xi)
    return h

# Base parameters (our "calibrated" values)
base_params = {
    'K': K_TRUE,      # 15.0 m/d
    'R': R_TRUE,      # 0.001 m/d
    'h0': H0,         # 100.0 m
    'h1': H1,         # 95.0 m
    'L': L,           # 1000.0 m
}

# Observation point for sensitivity (middle of domain)
x_sens = 500.0

# Calculate base case head
h_base = gw_model_full(**base_params, x=x_sens)
print(f"Base case head at x={x_sens}m: {h_base:.2f} m")
print(f"\nBase parameters:")
for name, val in base_params.items():
    print(f"  {name} = {val}")

## One-at-a-Time (OAT) Sensitivity Analysis

We vary each parameter by ±20% while holding others constant, and measure the change in predicted head at our observation point.

In [ ]:
def oat_sensitivity(base_params, param_name, variation_pct, x_obs):
    """
    Perform OAT sensitivity for one parameter.
    
    Returns: (h_low, h_base, h_high)
    """
    # Base case
    h_base = gw_model_full(**base_params, x=x_obs)
    
    # Low case (-variation_pct%)
    params_low = base_params.copy()
    params_low[param_name] = base_params[param_name] * (1 - variation_pct/100)
    h_low = gw_model_full(**params_low, x=x_obs)
    
    # High case (+variation_pct%)
    params_high = base_params.copy()
    params_high[param_name] = base_params[param_name] * (1 + variation_pct/100)
    h_high = gw_model_full(**params_high, x=x_obs)
    
    return h_low, h_base, h_high

# Parameter labels for display
param_labels = {
    'K': 'Hydraulic conductivity (K)',
    'R': 'Recharge (R)',
    'h0': 'Upstream head (h₀)',
    'h1': 'Downstream head (h₁)',
    'L': 'Domain length (L)',
}

variation = 20  # ±20%

# Calculate sensitivities
sensitivities = {}
print(f"Sensitivity analysis (±{variation}% variation):")
print("=" * 65)
print(f"{'Parameter':<30} {'h_low [m]':>10} {'h_high [m]':>10} {'Range [m]':>10}")
print("-" * 65)

for param in param_labels.keys():
    h_low, h_base, h_high = oat_sensitivity(base_params, param, variation, x_sens)
    sensitivities[param] = {
        'low': h_low,
        'base': h_base,
        'high': h_high,
        'range': abs(h_high - h_low),
        'delta_low': h_low - h_base,
        'delta_high': h_high - h_base,
    }
    print(f"{param_labels[param]:<30} {h_low:>10.2f} {h_high:>10.2f} {abs(h_high-h_low):>10.2f}")

print("-" * 65)

## Tornado Diagram

A **tornado diagram** visualizes parameter sensitivity by showing the range of model output caused by varying each parameter. Parameters are sorted by their influence (largest at top).

In [ ]:
def create_tornado_diagram(sensitivities, param_labels, h_base, variation, figsize=(8, 5)):
    """
    Create a tornado diagram showing parameter sensitivities.
    """
    # Sort parameters by total range (most sensitive at top)
    sorted_params = sorted(sensitivities.keys(), 
                          key=lambda p: abs(sensitivities[p]['range']))
    
    fig, ax = plt.subplots(figsize=figsize)
    
    y_positions = np.arange(len(sorted_params))
    bar_height = 0.6
    
    for i, param in enumerate(sorted_params):
        s = sensitivities[param]
        delta_low = s['delta_low']
        delta_high = s['delta_high']
        
        # Low side (what happens when we DECREASE parameter)
        color_low = '#E74C3C' if delta_low < 0 else '#3498DB'
        ax.barh(i, delta_low, height=bar_height, left=0,
               color=color_low, alpha=0.8, 
               edgecolor='darkred' if delta_low < 0 else 'darkblue', linewidth=1)
        
        # High side (what happens when we INCREASE parameter)
        color_high = '#3498DB' if delta_high > 0 else '#E74C3C'
        ax.barh(i, delta_high, height=bar_height, left=0,
               color=color_high, alpha=0.8,
               edgecolor='darkblue' if delta_high > 0 else 'darkred', linewidth=1)
    
    # Base line
    ax.axvline(x=0, color='black', linewidth=2, linestyle='-')
    
    # Labels
    ax.set_yticks(y_positions)
    ax.set_yticklabels([param_labels[p] for p in sorted_params])
    ax.set_xlabel('Change in predicted head [m]')
    ax.set_title(f'Parameter Sensitivity (±{variation}% variation)\nPrediction at x = {x_sens:.0f} m', 
                fontsize=12, fontweight='bold')
    
    # Add legend
    legend_elements = [
        Patch(facecolor='#3498DB', edgecolor='darkblue', label='Head increases'),
        Patch(facecolor='#E74C3C', edgecolor='darkred', label='Head decreases'),
    ]
    ax.legend(handles=legend_elements, loc='lower right')
    
    # Grid
    ax.grid(axis='x', alpha=0.3)
    ax.set_axisbelow(True)
    
    plt.tight_layout()
    return fig, ax

fig, ax = create_tornado_diagram(sensitivities, param_labels, h_base, variation)
plt.show()

## Interpretation

The tornado diagram reveals:

1. **Most sensitive parameters** (top of diagram) - these control model predictions
2. **Least sensitive parameters** (bottom) - these have little effect on output
3. **Direction of effect** - does increasing the parameter increase or decrease head?

### Physical interpretation:

- **Boundary heads (h₀, h₁)**: Direct, linear effect on predictions
- **Recharge (R)**: Increases head (more water in)
- **Hydraulic conductivity (K)**: *Decreases* head (water flows out faster)
- **Domain length (L)**: Complex effect depending on location

### Implications for modelling:

- **Calibration**: Focus on sensitive parameters; insensitive ones cannot be reliably estimated
- **Uncertainty**: Uncertainty in sensitive parameters matters most for predictions
- **Data collection**: Prioritize measurements that constrain sensitive parameters

## Animated Tornado Diagram

Create an animation that builds up the tornado diagram parameter by parameter.

In [ ]:
def create_tornado_animation(sensitivities, param_labels, h_base, variation, figsize=(8, 5)):
    """
    Create animated tornado diagram that builds up bar by bar.
    """
    # Sort parameters by total range (least sensitive first for animation)
    sorted_params = sorted(sensitivities.keys(), 
                          key=lambda p: abs(sensitivities[p]['range']))
    
    fig, ax = plt.subplots(figsize=figsize)
    
    n_params = len(sorted_params)
    y_positions = np.arange(n_params)
    bar_height = 0.6
    
    # Calculate x-axis limits
    all_deltas = []
    for param in sorted_params:
        all_deltas.extend([sensitivities[param]['delta_low'], 
                          sensitivities[param]['delta_high']])
    x_min, x_max = min(all_deltas) * 1.3, max(all_deltas) * 1.3
    
    def animate(frame):
        ax.clear()
        ax.set_xlim(x_min, x_max)
        ax.set_ylim(-0.5, n_params - 0.5)
        ax.axvline(x=0, color='black', linewidth=2, linestyle='-')
        ax.set_yticks(y_positions)
        ax.set_yticklabels([param_labels[p] for p in sorted_params])
        ax.set_xlabel('Change in predicted head [m]')
        ax.set_title(f'Parameter Sensitivity (±{variation}% variation)\nPrediction at x = {x_sens:.0f} m',
                    fontsize=12, fontweight='bold')
        ax.grid(axis='x', alpha=0.3)
        ax.set_axisbelow(True)
        
        # Draw bars up to current frame
        n_bars = min(frame + 1, n_params)
        
        for i in range(n_bars):
            param = sorted_params[i]
            s = sensitivities[param]
            delta_low = s['delta_low']
            delta_high = s['delta_high']
            
            # Low side
            color_low = '#E74C3C' if delta_low < 0 else '#3498DB'
            ax.barh(i, delta_low, height=bar_height, left=0,
                   color=color_low, alpha=0.8, 
                   edgecolor='darkred' if delta_low < 0 else 'darkblue', linewidth=1)
            
            # High side  
            color_high = '#3498DB' if delta_high > 0 else '#E74C3C'
            ax.barh(i, delta_high, height=bar_height, left=0,
                   color=color_high, alpha=0.8,
                   edgecolor='darkblue' if delta_high > 0 else 'darkred', linewidth=1)
        
        # Highlight current parameter being tested
        if frame < n_params:
            current_param = sorted_params[frame]
            ax.annotate(f'Testing: {param_labels[current_param]}',
                       xy=(0, frame), xytext=(x_max * 0.4, frame + 0.3),
                       fontsize=10, color='green', fontweight='bold',
                       ha='center')
        
        # Add legend on final frames
        if frame >= n_params - 1:
            legend_elements = [
                Patch(facecolor='#3498DB', edgecolor='darkblue', label='Head increases'),
                Patch(facecolor='#E74C3C', edgecolor='darkred', label='Head decreases'),
            ]
            ax.legend(handles=legend_elements, loc='lower right', fontsize=9)
            
            # Highlight most sensitive
            most_sensitive = sorted_params[-1]
            ax.text(0.02, 0.98, f'Most sensitive: {param_labels[most_sensitive]}',
                   transform=ax.transAxes, fontsize=10, fontweight='bold',
                   verticalalignment='top', color='darkgreen')
        
        plt.tight_layout()
        return []
    
    # Create animation: one frame per parameter + hold frames at end
    n_frames = n_params + 3
    anim = animation.FuncAnimation(fig, animate, frames=n_frames, interval=800, blit=True)
    plt.close(fig)
    return anim, fig

anim, fig = create_tornado_animation(sensitivities, param_labels, h_base, variation)
HTML(anim.to_jshtml())

## Save Animation as GIF

In [ ]:
# Recreate animation for saving
anim, fig = create_tornado_animation(sensitivities, param_labels, h_base, variation, figsize=(9, 5))

# Save as GIF
gif_path = 'sensitivity_tornado.gif'
anim.save(gif_path, writer='pillow', fps=1.2, dpi=100)
print(f"Animation saved to {gif_path}")

# Display file size
import os
size_kb = os.path.getsize(gif_path) / 1024
print(f"File size: {size_kb:.0f} KB")

## Key Takeaways

1. **Sensitivity analysis reveals which parameters matter** - not all parameters equally influence predictions

2. **Insensitive parameters cannot be calibrated** - if the model doesn't respond to a parameter, observations cannot constrain it

3. **Focus uncertainty analysis on sensitive parameters** - uncertainty in insensitive parameters doesn't affect predictions much

4. **Limitations of OAT analysis**:
   - Assumes parameters vary independently (misses interactions)
   - Tests only ±X% variation (may miss nonlinear effects)
   - More rigorous: Global sensitivity analysis (Morris, Sobol methods)